In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

In [3]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
    
DATA_PATH = PROJECT_ROOT / "data" / "skyCity Auckland Restaurants & Bars.csv"

df = pd.read_csv(DATA_PATH)

In [4]:
df.head()

,CuisineType,RestaurantID,RestaurantName,Segment,Subregion,GrowthFactor,AOV,MonthlyOrders,InStoreOrders,InStoreRevenue,UberEatsOrders,DoorDashOrders,SelfDeliveryOrders,UberEatsRevenue,DoorDashRevenue,SelfDeliveryRevenue,COGSRate,OPEXRate,CommissionRate,DeliveryRadiusKM,DeliveryCostPerOrder,SD_DeliveryTotalCost,InStoreNetProfit,UberEatsNetProfit,DoorDashNetProfit,SelfDeliveryNetProfit,InStoreShare,UE_share,DD_share,SD_share
0,Burgers,25731,Urban Burgers House,Cafe,North Shore,1.03,43.97,668,197,8662.09,212,118,141,9321.64,5188.46,6199.77,0.21,0.37,0.28,11,3.25,458.25,3682.14,1352.45,752.78,2177.19,0.42,0.45,0.25,0.30
1,Burgers,25123,Urban Burgers Diner,QSR,South Auckland,1.05,40.45,1388,259,10476.55,508,282,339,20548.60,11406.90,13712.55,0.25,0.41,0.28,16,4.72,1600.08,3605.72,1318.61,731.99,3119.38,0.23,0.45,0.25,0.30
2,Burgers,25177,King Burgers Eatery,Cafe,West Auckland,1.04,40.03,1717,524,20975.72,537,298,358,21496.11,11928.94,14330.74,0.22,0.41,0.30,11,3.25,1163.50,7810.95,1555.90,863.42,4172.99,0.44,0.45,0.25,0.30
3,Burgers,25540,Classic Burgers Tavern,QSR,North Shore,1.03,36.28,1083,216,7836.48,390,217,260,14149.20,7872.76,9432.80,0.24,0.44,0.33,3,0.89,231.40,2546.02,-72.25,-40.20,2833.26,0.25,0.45,0.25,0.30
4,Burgers,25258,Lucky Burgers Bistro,Cafe,South Auckland,1.05,34.34,1230,261,8962.74,436,242,291,14972.24,8310.28,9992.94,0.22,0.43,0.33,9,2.66,774.06,3093.09,226.17,125.53,2674.56,0.27,0.45,0.25,0.30


In [5]:
df.shape

(1696, 30)

In [6]:
revenue_cols = [
    "InStoreRevenue",
    "UberEatsRevenue",
    "DoorDashRevenue",
    "SelfDeliveryRevenue"
]

order_cols = [
    "InStoreOrders",
    "UberEatsOrders",
    "DoorDashOrders",
    "SelfDeliveryOrders"
]

profit_cols = [
    "InStoreNetProfit",
    "UberEatsNetProfit",
    "DoorDashNetProfit",
    "SelfDeliveryNetProfit"
]

df["TotalRevenue"] = df[revenue_cols].sum(axis=1)
df["TotalOrdersCalculated"] = df[order_cols].sum(axis=1)
df["TotalNetProfit"] = df[profit_cols].sum(axis=1)

assert (df["TotalOrdersCalculated"] == df["MonthlyOrders"]).all(), \
    "Some channel order counts do not match MonthlyOrders."

df["OverallProfitMargin"] = np.where(
    df["TotalRevenue"] > 0,
    df["TotalNetProfit"] / df["TotalRevenue"] * 100,
    np.nan
)

df["ProfitPerOrder"] = np.where(
    df["TotalOrdersCalculated"] > 0,
    df["TotalNetProfit"] / df["TotalOrdersCalculated"],
    np.nan
)

display(
    df[[
        "RestaurantID",
        "TotalRevenue",
        "TotalOrdersCalculated",
        "TotalNetProfit",
        "OverallProfitMargin",
        "ProfitPerOrder"
    ]].head(20)
)

,RestaurantID,TotalRevenue,TotalOrdersCalculated,TotalNetProfit,OverallProfitMargin,ProfitPerOrder
0,25731,29371.96,668,7964.56,27.12,11.92
1,25123,56144.60,1388,8775.70,15.63,6.32
2,25177,68731.51,1717,14403.26,20.96,8.39
3,25540,39291.24,1083,5266.83,13.40,4.86
4,25258,42238.20,1230,6119.35,14.49,4.98
5,26138,30499.76,827,4582.50,15.02,5.54
6,25396,40807.92,881,9608.97,23.55,10.91
7,25087,27792.00,800,5056.68,18.19,6.32
8,25125,20067.48,612,3426.42,17.07,5.60
9,25762,20632.04,572,3748.22,18.17,6.55


In [7]:
channel_order_mapping = {
    "InStore": "InStoreOrders",
    "UberEats": "UberEatsOrders",
    "DoorDash": "DoorDashOrders",
    "SelfDelivery": "SelfDeliveryOrders"
}

for channel, order_col in channel_order_mapping.items():
    df[f"{channel}OrderMix"] = np.where(
        df["MonthlyOrders"] > 0,
        df[order_col] / df["MonthlyOrders"],
        np.nan
    )
    
df["DeliveryOrders"] = df[
    ["UberEatsOrders", "DoorDashOrders", "SelfDeliveryOrders"]
].sum(axis=1)

df["DeliveryOrderMix"] = np.where(
    df["MonthlyOrders"] > 0,
    df["DeliveryOrders"] / df["MonthlyOrders"],
    np.nan
)

display(
    df[[
        "RestaurantName",
        "InStoreOrderMix",
        "UberEatsOrderMix",
        "DoorDashOrderMix",
        "SelfDeliveryOrderMix",
        "DeliveryOrderMix"
    ]].head(10)
)

,RestaurantName,InStoreOrderMix,UberEatsOrderMix,DoorDashOrderMix,SelfDeliveryOrderMix,DeliveryOrderMix
0,Urban Burgers House,0.29,0.32,0.18,0.21,0.71
1,Urban Burgers Diner,0.19,0.37,0.20,0.24,0.81
2,King Burgers Eatery,0.31,0.31,0.17,0.21,0.69
3,Classic Burgers Tavern,0.20,0.36,0.20,0.24,0.80
4,Lucky Burgers Bistro,0.21,0.35,0.20,0.24,0.79
5,King Burgers Kitchen,0.19,0.36,0.20,0.24,0.81
6,Urban Burgers Eatery,0.20,0.36,0.20,0.24,0.80
7,Urban Burgers Tavern,0.09,0.41,0.23,0.27,0.91
8,Classic Burgers Tavern,0.25,0.34,0.19,0.23,0.75
9,Lucky Burgers Cafe,0.22,0.35,0.19,0.23,0.78


In [8]:
df["TotalNetProfit"] = df[profit_cols].sum(axis=1)

print("Target summary:")
display(df["TotalNetProfit"].describe())

print("Missing Target values:", df["TotalNetProfit"].isna().sum())
print("Infinte Target values:", np.isinf(df["TotalNetProfit"]).sum())

print("Loss-making restaurants:", (df["TotalNetProfit"] < 0).sum())
print("Profitable restaurants:", (df["TotalNetProfit"] > 0).sum())

Target summary:


count     1696.00
mean      4638.15
std       5829.60
min     -10192.83
25%        556.61
50%       4932.86
75%       8419.29
max      27368.36
Name: TotalNetProfit, dtype: float64

Missing Target values: 0
Infinte Target values: 0
Loss-making restaurants: 408
Profitable restaurants: 1288


In [9]:
target = "TotalNetProfit"

categorical_features = [
    "CuisineType",
    "Segment",
    "Subregion"
]

numeric_features = [
    "GrowthFactor",
    "AOV",
    "MonthlyOrders",
    "InStoreOrders",
    "UberEatsOrders",
    "DoorDashOrders",
    "SelfDeliveryOrders",
    "COGSRate",
    "OPEXRate",
    "CommissionRate",
    "DeliveryRadiusKM",
    "DeliveryCostPerOrder"
]

feature_columns = categorical_features + numeric_features

X = df[feature_columns].copy()
y = df[target].copy()

print("Number of input features:", len(feature_columns))
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

display(X.head())

Number of input features: 15
Feature matrix shape: (1696, 15)
Target shape: (1696,)


,CuisineType,Segment,Subregion,GrowthFactor,AOV,MonthlyOrders,InStoreOrders,UberEatsOrders,DoorDashOrders,SelfDeliveryOrders,COGSRate,OPEXRate,CommissionRate,DeliveryRadiusKM,DeliveryCostPerOrder
0,Burgers,Cafe,North Shore,1.03,43.97,668,197,212,118,141,0.21,0.37,0.28,11,3.25
1,Burgers,QSR,South Auckland,1.05,40.45,1388,259,508,282,339,0.25,0.41,0.28,16,4.72
2,Burgers,Cafe,West Auckland,1.04,40.03,1717,524,537,298,358,0.22,0.41,0.30,11,3.25
3,Burgers,QSR,North Shore,1.03,36.28,1083,216,390,217,260,0.24,0.44,0.33,3,0.89
4,Burgers,Cafe,South Auckland,1.05,34.34,1230,261,436,242,291,0.22,0.43,0.33,9,2.66


In [10]:
print("Missing vlaues in features:", X.isna().sum().sum())
print("Missing target values:", y.isna().sum())

print("\nFeature data types:")
display(X.dtypes.to_frame("Data Type"))

print("\nTarget statistics:")
display(y.describe())

Missing vlaues in features: 0
Missing target values: 0

Feature data types:


,Data Type
CuisineType,str
Segment,str
Subregion,str
GrowthFactor,float64
AOV,float64
MonthlyOrders,int64
InStoreOrders,int64
UberEatsOrders,int64
DoorDashOrders,int64
SelfDeliveryOrders,int64



Target statistics:


count     1696.00
mean      4638.15
std       5829.60
min     -10192.83
25%        556.61
50%       4932.86
75%       8419.29
max      27368.36
Name: TotalNetProfit, dtype: float64

In [14]:
correlation_data = X[numeric_features].copy()
correlation_data["TotalNetProfit"] = y

correlation_matrix = correlation_data.corr(numeric_only=True)

display(
    correlation_matrix[["TotalNetProfit"]]
    .sort_values("TotalNetProfit", ascending=False)
)

,TotalNetProfit
TotalNetProfit,1.00
SelfDeliveryOrders,0.38
AOV,0.17
MonthlyOrders,0.17
DoorDashOrders,0.16
UberEatsOrders,0.16
GrowthFactor,0.06
CommissionRate,-0.05
DeliveryRadiusKM,-0.07
DeliveryCostPerOrder,-0.07


In [15]:
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "processed"

modeling_df = X.copy()
modeling_df[target] = y

modeling_df.to_csv(
    OUTPUT_DIR / "restaurant_modeling_dataset.csv", index=False
)